# 01 — Data Understanding

This notebook performs only Phase 1 inspection on the fixed Kaggle CSV. It discovers the file and schema at runtime, reports data-quality basics, shortlists purchase/outcome targets, and identifies likely post-event leakage. No preprocessing or modelling is performed.

## Setup and load the actual CSV

The notebook imports the reusable functions from `src/data_prep.py`, keeping notebook and script behaviour in sync. The fixed seed makes the displayed sample reproducible.

In [1]:
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_prep import (
    RANDOM_SEED, choose_target, inspect_dataset, load_dataset,
    render_summary, scan_likely_leakage, shortlist_target_candidates, write_summary
)

np.random.seed(RANDOM_SEED)
df, csv_path = load_dataset(PROJECT_ROOT / 'data')
print(f'Loaded actual CSV: {csv_path.resolve()}')

Loaded actual CSV: P:\College\Sem VII Acad\IPRM\Project\data\Ecommerce.csv


## Structural and data-quality inspection

Shape and column types establish what is actually present. Missing counts and duplicate rows reveal immediate quality issues. A seeded five-row sample provides a compact, repeatable view of real records.

In [2]:
inspection = inspect_dataset(df)
print('Shape:', inspection['shape'])
print('\nColumn names:')
print(inspection['columns'])
print('\nDtypes:')
print(inspection['dtypes'].to_string())
print('\nMissing values per column:')
print(inspection['missing_counts'].to_string())
print('\nDuplicate row count:', inspection['duplicate_rows'])
print('\nReproducible 5-row sample:')
print(inspection['sample'].to_string(index=False))

Shape: (25000, 29)

Column names:
['customer_id', 'session_id', 'visit_date', 'device_type', 'user_type', 'marketing_channel', 'product_id', 'product_category', 'unit_price', 'quantity', 'discount_percent', 'discount_amount', 'revenue', 'pages_viewed', 'time_on_site_sec', 'added_to_cart', 'purchased', 'cart_abandoned', 'rating', 'review_text', 'review_helpful_votes', 'payment_method', 'visit_day', 'visit_month', 'visit_weekday', 'visit_season', 'session_duration_bucket', 'revenue_normalized', 'location']

Dtypes:
customer_id                  int64
session_id                   int64
visit_date                  object
device_type                  int64
user_type                    int64
marketing_channel            int64
product_id                   int64
product_category             int64
unit_price                 float64
quantity                     int64
discount_percent             int64
discount_amount            float64
revenue                    float64
pages_viewed              

## Programmatic target shortlist

A defensible classification target should describe a purchase, conversion, or order outcome and have a binary/categorical dtype. The code scores only columns that satisfy both conditions and chooses the strongest match; it does not assume a dataset-specific target name.

In [3]:
candidates = shortlist_target_candidates(df)
print('Candidate target columns and reasoning:')
for candidate in candidates:
    print(f"- {candidate['column']} [score={candidate['score']}]: {candidate['reason']}")

target, target_reason = choose_target(candidates)
print(f'\nChosen target: {target}')
print('Reason:', target_reason)

Candidate target columns and reasoning:
- purchased [score=9]: binary field with 2 non-null values; name matched outcome term(s): purchase, purchased
- cart_abandoned [score=6]: binary field with 2 non-null values; name matched outcome term(s): abandon, abandoned

Chosen target: purchased
Reason: highest programmatic relevance score (9); binary field with 2 non-null values; name matched outcome term(s): purchase, purchased


## Likely target leakage

For a model intended to predict the selected outcome before it happens, fields created at checkout or after the session outcome would make evaluation unrealistic. A conservative semantic scan records each proposed exclusion and its reason.

In [4]:
leakage = scan_likely_leakage(df, target)
print('Likely leakage exclusions:')
for item in leakage:
    print(f"- {item['column']}: {item['reason']}")

Likely leakage exclusions:
- revenue: post-purchase monetary outcome; its value is produced by a completed transaction (matched: revenue)
- cart_abandoned: competing end-of-session outcome; it directly reveals how the session ended (matched: abandon, abandoned)
- rating: post-purchase feedback; it can only be recorded after the customer outcome (matched: rating)
- review_text: post-purchase feedback; it can only be recorded after the customer outcome (matched: review)
- review_helpful_votes: post-purchase feedback; it can only be recorded after the customer outcome (matched: helpful_vote, helpful_votes, review)
- payment_method: checkout/payment-stage information; conservatively unavailable before the purchase decision (matched: payment, payment_method)
- revenue_normalized: post-purchase monetary outcome; its value is produced by a completed transaction (matched: revenue)


## Save the Phase 1 result

The same observed details and reasoning are written to `results/dataset_summary.txt` for reproducibility.

In [5]:
summary = render_summary(csv_path, inspection, candidates, target, target_reason, leakage)
summary_path = write_summary(summary, PROJECT_ROOT / 'results' / 'dataset_summary.txt')
print(f'Saved complete Phase 1 summary to: {summary_path.resolve()}')

Saved complete Phase 1 summary to: P:\College\Sem VII Acad\IPRM\Project\results\dataset_summary.txt


## Phase 1 boundary

Inspection is complete. Preprocessing, exploratory plots, model training, evaluation, and explainability are intentionally deferred to later phases.